In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import random
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import dash
import dash_core_components as dcc
import dash_html_components as html
from dash.dependencies import Output, Input, State
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn import svms
from sklearn.metrics import root_mean_squared_error
from sklearn.neighbors import KNeighborsRegressor


df_original = sns.load_dataset("iris")

df = df_original.copy()

# codificando las variables nominales para el algoritmo

df["species_encoded"] = LabelEncoder().fit_transform(df["species"])

pca_model = PCA(n_components=2) 

pca = pca_model.fit_transform(df[df.columns[:4]])

df["PCA_1"] = pca.T[0]
df["PCA_2"] = pca.T[1]

df

: 

In [ ]:
var = df.loc[(df["species"] == "virginica") | (df["species"] == "setosa"), ["PCA_1","PCA_2"]]
var["PCA_1"] = round(var["PCA_1"]*100).astype(int)
var["PCA_2"] = round(var["PCA_2"]*100).astype(int)

def add_values(length, value_min, value_max):
    new_values = np.array([])
    for i in range(length):
        value = random.randint(value_min, value_max)
        new_values = np.append(new_values, value)
    return new_values
    
grp1_pca_1 = add_values(len(var)/2, 100, 400)
grp1_pca_2 = add_values(len(var)/2, -100, 100)
grp2_pca_1 = add_values(len(var)*1.5, -400, -200)
grp2_pca_2 = add_values(len(var)*1.5, -100, 100)

new_data_1 = np.array([grp1_pca_1,grp1_pca_2]).T
new_data_2 = np.array([grp2_pca_1,grp2_pca_2]).T

df_new_data_1 = pd.DataFrame(new_data_1, columns=["PCA_1","PCA_2"])
df_new_data_2 = pd.DataFrame(new_data_2, columns=["PCA_1","PCA_2"])

var = pd.concat([var, df_new_data_1, df_new_data_2])

plt.scatter(var["PCA_2"], var["PCA_1"])
plt.grid("on")
plt.show()
print(var.shape)

: 

#### Feature Engireering
Es el proceso de seleccionar, eliminar y transformar las variables de un conjunto para generar una entrada de datos eficaz en un modelo específico, una popular técnica es el Análisis de Componentes Principales(PCA), cuyo fundamento se basa en conceptos de álgebra lineal y su objetivo en la reducción de la dimensionalidad pero conservando a su vez la mayor cantidad de información, proyectando una especie de sombra en los datos. Se implementa a la hora de combatir la maldición de la dimensionalidad, permitir crear gráficos cartesianos y estimular a algoritmos que no sean robustos, como en este ejemplo.

In [ ]:
SVM = svm.SVC(kernel="rbf", C=1, probability=True)

SVM.fit(pca, df["species_encoded"])

x_min, x_max = df["PCA_1"].min() - 1, df["PCA_1"].max() + 1
y_min, y_max = df["PCA_2"].min() - 1, df["PCA_2"].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1), np.arange(y_min, y_max, 0.1))

Z = SVM.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.contourf(xx, yy, Z, alpha=0.8)
plt.scatter(df["PCA_1"], df["PCA_2"], c=df["species_encoded"], s=20, edgecolor='k')
plt.xlabel('Característica 1')
plt.ylabel('Característica 2')
plt.title('Ajuste del modelo SVM con kernel radial básico y C=1')
plt.show()

#### Sobreajuste
Es en un de los principales y más comunes problemas a la hora de desarrollar un modelo predictivo, siendo basicamente elsobreentrenar y, debido a esto, otorgar un precisión excesiva sobre los datos de entrenamiento provocando que el modelo no esté preparado para adaptarse a nuevas situaciones. 

In [ ]:
object = [[df["sepal_length"].mean(), df["sepal_width"].mean(), df["petal_length"].mean(), df["petal_width"].mean()]]

pca_object = pca_model.transform(object)

# prediciendo la clase del nuevo objeto

predict_encoded = SVM.predict(pca_object)

# asociando las variables codificadas con sus versiones originales

classes = list(zip(df["species"].unique(),df["species_encoded"].unique()))

# asociando la predicción a su clase

predict_example = classes[predict_encoded[0]][0]

probability = SVM.predict_proba(pca_object)

probability = probability[0, predict_encoded]*100

probability = str(probability[0])

print("--------------------------------------------")
print("Nuevo objeto \n")
for c in df.columns[:4]:
    print(f"{c}: {df[c].mean()}")
    
print(f"\npredicción: {predict_example} | probabilidad: {probability[:4]}%")
print("--------------------------------------------")

#### Dashboard que refleja la transformación de las varibles y permite introduir datos 


In [ ]:
app = dash.Dash(__name__)

graph_pca = go.Figure()
graph_pca.add_trace(go.Scatter(x=df["PCA_1"],y=df["PCA_2"],mode="markers",marker_color="blue",name="especies"))
graph_pca.update_layout(title="Figura PCA(principal components analysis)")
graph_pca.update_layout(legend=dict(font=dict(size=9)))

predict_text = html.B(children=[],id="predict",style={})
probability_text =  html.B(children=[],id="probability")

colors_species = {
    "setosa":"blue",
    "virginica":"green",
    "versicolor":"red"
}

app.layout =  html.Div(id="body",className="e4_body",children=[
    html.H1("Clasificador de especies",id="title",className="e4_title"),
    html.Div(id="dashboard",className="e4_dashboard",children=[
        html.Div(className="e4_graph_div",children=[
            dcc.Checklist(id="checklist",style={"display":"flex","margin-top":"30px"},
                        options=[
                            {"label":i,"value":i} for i in df_original.columns[:-1]
                        ],
                        value=["sepal_length","sepal_width","petal_length"]),
            dcc.Graph(id="graph-1",className="e4_graph",figure={})
        ]),
        html.Div(className="e4_graph_div",children=[
            dcc.Graph(id="graph-2",className="e4_graph",figure=graph_pca),
            html.Form(id="input_div",className="input_div",children=[
                dcc.Input(id="input_1",className="input",type="text",placeholder="sepal_length",size="7"),
                dcc.Input(id="input_2",className="input",type="text",placeholder="sepal_width",size="7"),
                dcc.Input(id="input_3",className="input",type="text",placeholder="petal_length",size="7"),
                dcc.Input(id="input_4",className="input",type="text",placeholder="petal_width",size="7"),
                html.Button("enviar",id="button",type="button",className="input",n_clicks=0)
            ]),
            html.P(["predicción: ",predict_text," | probabildad: ",probability_text,"%"],className="e4_predict")
        ])
    ])
])
    
@app.callback(
    Output(component_id="graph-1",component_property="figure"),
    [Input(component_id="checklist",component_property="value")]
)

def update_graph(slct_var):
    if len(slct_var) != 3:
        return {"data":[]}
    
    graph_multi = px.scatter_3d(df, x=slct_var[0], y=slct_var[1], z=slct_var[2], color='species')
    graph_multi.update_layout(title="Figura tridimensional")
    graph_multi.update_layout(legend=dict(font=dict(size=9.5)))
    
    return graph_multi
    
@app.callback(
    [Output(component_id="graph-2",component_property="figure"),
    Output(component_id="predict",component_property="children"),
    Output(component_id="probability",component_property="children"),
    Output(component_id="predict",component_property="style"),
    Output(component_id="probability",component_property="style")],
    [Input(component_id="button",component_property="n_clicks")],
    [State(component_id="input_1",component_property="value"),
    State(component_id="input_2",component_property="value"),
    State(component_id="input_3",component_property="value"),
    State(component_id="input_4",component_property="value")]
)

def update_graph(n_clicks, var_1, var_2, var_3, var_4):
    if n_clicks is not None and n_clicks > 0:
        object = [[var_1,var_2,var_3,var_4]]    
        pca_object = pca_model.transform(object)
        predict_encoded = SVM.predict(pca_object)
        predict = classes[predict_encoded[0]][0]
        predict_color = {"color":colors_species[predict]}
        probability_color = {"color":colors_species[predict]}
        probability = SVM.predict_proba(pca_object)
        probability = probability[0,predict_encoded] * 100
        probability = str(probability[0])
        probability = probability[:4]
        graph_pca.add_trace(go.Scatter(x=[pca_object[0,0]], y=[pca_object[0,1]], mode="markers", marker_color="red", name=f"nueva especie({predict})"))
    
    return graph_pca, predict, probability, predict_color, probability_color


if __name__ == "__main__":
    app.run_server(debug=False)

#### Machine Learning Deployment
Hasta ahora hemos generado modelos predictivos que se encargan de resolver problemas asignados pero no fueron llevados a producción, es decir, tomar el modelo en particular desarrollado a partir de los fundamentos de Machine Learning, ponerlo a disposición de un usuario final y lograr que este modelo se pueda actualizar periódicamente para garantizar que tenga siempre el desempeño esperado. Para esto se requieren de habilidades menos científicas y más propiamente de un Ingenirero de software, donde se implementan herramientas como Docker que ayuda a crear y distribuir aplicaciones mediante contenedores que incluyan las librerías específicas instaladas, habiliten puertos, contengan volumes, etc y luego del uso de un orquestador como Kubernetes donde se gestionarán los recursos en los contenedores, servicios, configuraciones y la infraestructura en general del despliegue que se realizará. Como ya habíamos mencionado, esta área es un híbrido entre la ciencia de datos que abarca procesos como la extracción de datos, el análisis exploratorio, el preprocesamiento de datos, el entrenamiento de modelos y su evaluación y un ingeniero de softwre convencional, sin embargo, en el Machine Learning Deployment el software es mayormente dinámico, ya que el constante cambio en la información provoca que se requiera de fases posteirores al despliegue y puesta en servicio denominadas monitoreo y mantenimiento, el objetivo en estas fases es la verificación del desempeño del modelo y la repetición de las fases anteriormente mencionadas en el caso de que haya habido una degradación.